# Азы работы с файлами в Python

## Варианты задания путей к файлам

### Через строки

In [ ]:
# ПЛОХО: \n и \t превращаются в специальные символы
bad_path = "C:\new\test.txt"
print(bad_path)

# ХОРОШО: удваиваем обратные слеши
path = "C:\\new\\test.txt"
print(path)

# ПРОЩЕ: используем raw-строку с префиксом r
path = r"C:\new\test.txt"
print(path)

In [ ]:
# Linux / macOS: обратных слешей нет
path = "/home/user/data/test.txt"
print(path)

# Относительный путь
path = "data/test.txt"

In [ ]:
# Собираем путь из строк
folder = "data"
filename = "test.txt"
path = folder + "/" + filename
print(path)

# То же самое через f-строку
path = f"{folder}/{filename}"
print(path)

### Через стандартный пакет `os`

In [ ]:
import os

In [ ]:
# Текущая рабочая папка
print(os.getcwd())

In [ ]:
# Собрать путь к папке и файлу
folder = os.path.join("data", "texts")
path = os.path.join(folder, "example.txt")

print(folder)
print(path)

In [ ]:
# Создать папку, если её нет
os.makedirs(folder, exist_ok=True)

In [ ]:
# Части пути
print(os.path.basename(path))  # Имя файла: example.txt
print(os.path.dirname(path))   # Папка: data/texts
print(os.path.splitext(path))  # Путь без расширения и расширение

In [ ]:
# Полный путь
print(os.path.abspath(path))

In [ ]:
# Записать файл
with open(path, "w", encoding="utf-8") as file:
    file.write("Привет!")

# Проверить существование
print(os.path.exists(path))    # Существует ли путь
print(os.path.isfile(path))    # Это файл?
print(os.path.isdir(folder))  # Это папка?

### Относительные пути
```
project/
├── config.json
├── data/
│   └── points.csv
└── notebooks/          ← текущая рабочая папка
    ├── lesson.ipynb
    ├── notes.txt
    └── results/
        └── report.txt
```

#### Относительные пути для MacOS / Linux

In [ ]:
# Текущая папка
path = "."
path = "./"

# Файл в текущей папке — два равнозначных варианта
path = "notes.txt"
path = "./notes.txt"

# Вложенная папка
path = "./results/report.txt"   # project/notebooks/results/report.txt

# На один уровень выше
path = ".."                    # project/
path = "../config.json"        # project/config.json

# Вверх, затем в соседнюю папку
path = "../data/points.csv"     # project/data/points.csv

# На два уровня выше
path = "../.."                 # папка, содержащая project/
path = "../../other/file.txt"  # соседний с project каталог other/

# Чтение файла из соседней папки
with open("../data/points.csv", "r", encoding="utf-8") as file:
    text = file.read()

#### Те же пути с обратными слешами Windows

In [ ]:
# ПРАВИЛЬНО: raw-строки или удвоенные слеши
path = r".\results\report.txt"
path = r"..\data\points.csv"
path = "..\\data\\points.csv"

# ПЛОХО: \n — перевод строки, \t — табуляция
bad_path = "..\new\test.txt"
print(bad_path)

# ПРАВИЛЬНО
path = r"..\new\test.txt"

### Через пакет `pathlib`

In [ ]:
from pathlib import Path

In [ ]:
# Текущая рабочая папка
print(Path.cwd())

In [ ]:
# Путь к папке и файлу
folder = Path("data") / "texts"
path = folder / "example.txt"

print(folder)
print(path)

In [ ]:
# Создать папку вместе с недостающими родительскими папками
folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Части пути
print(path.name)    # example.txt — имя файла
print(path.stem)    # example — имя без расширения
print(path.suffix)  # .txt — расширение
print(path.parent)  # data/texts — родительская папка

In [ ]:
# Записать файл по этому пути
with open(path, "w", encoding="utf-8") as file:
    file.write("Привет!")

# Проверить существование
print(path.exists())   # Существует ли путь
print(path.is_file())  # Это файл?
print(folder.is_dir()) # Это папка?

In [ ]:
# Найти текстовые файлы в папке
for item in folder.glob("*.txt"):
    print(item)

## 1. Подготовка папки

`Path` помогает записывать пути одинаково на Windows, macOS и Linux. Если папки ещё нет, создаём её.

In [ ]:
from pathlib import Path

folder = Path("file_examples")
folder.mkdir(exist_ok=True)

## 2. Открытие без менеджера контекста

Так делать неудобно: файл приходится закрывать вручную. Пока не вызван `close()`, он остаётся открытым.

Если забыть `close()` или выполнение остановится до этой строки, файл может остаться открытым. В примере ниже мы не вызываем ошибку: просто смотрим, что файл ещё открыт, а затем закрываем его.

In [ ]:
file = open(folder / "manual.txt", "w", encoding="utf-8")
file.write("Файл открыт вручную\n")

print("Файл закрыт:", file.closed)

file.close()
print("После close():", file.closed)

## 3. Открытие через `with`

Внутри блока `with` работаем с файлом. При выходе из блока он закрывается автоматически. Отдельный `close()` не нужен.

`encoding="utf-8"` указывает кодировку текста.

In [ ]:
with open(folder / "example.txt", "w", encoding="utf-8") as file:
    file.write("Первая строка\n")
    print("Внутри with файл закрыт:", file.closed)

print("После with файл закрыт:", file.closed)

## 4. Чтение всего файла

Режим `r` — чтение. Метод `read()` читает весь оставшийся текст.

Сначала сохраняем текст в переменную, затем выводим его. Переменная остаётся доступна после закрытия файла.

In [ ]:
with open(folder / "example.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text)

## 5. Перезапись файла

Режим `w` создаёт файл, если его нет. Если файл существует, старое содержимое стирается уже при открытии.

Менеджер контекста не защищает от перезаписи: режим нужно выбирать внимательно.

In [ ]:
with open(folder / "example.txt", "w", encoding="utf-8") as file:
    file.write("Новый текст вместо старого\n")

with open(folder / "example.txt", "r", encoding="utf-8") as file:
    print(file.read())

## 6. Добавление в конец

Режим `a` добавляет текст в конец файла, сохраняя прежнее содержимое.

Если запустить только эту ячейку ещё раз, строка добавится повторно.

In [ ]:
with open(folder / "example.txt", "a", encoding="utf-8") as file:
    file.write("Добавленная строка\n")

with open(folder / "example.txt", "r", encoding="utf-8") as file:
    print(file.read())

## 7. Запись нескольких строк

`write()` не добавляет перевод строки автоматически. Поэтому указываем `\n` в конце каждой строки.

In [ ]:
lines = ["Первая строка", "Вторая строка", "Третья строка"]

with open(folder / "lines.txt", "w", encoding="utf-8") as file:
    for line in lines:
        file.write(line + "\n")

## 8. Построчное чтение

Цикл `for` читает строки по очереди. В строках уже есть `\n`, поэтому используем `end=""`, чтобы `print()` не добавлял ещё один перевод строки.

In [ ]:
with open(folder / "lines.txt", "r", encoding="utf-8") as file:
    for line in file:
        print(line, end="")

## 9. Чтение одной строки

Каждый вызов `readline()` читает следующую строку. В конце файла возвращается пустая строка `""`.

In [ ]:
with open(folder / "lines.txt", "r", encoding="utf-8") as file:
    first = file.readline()
    second = file.readline()

print(first, end="")
print(second, end="")

## 10. Чтение в список

`readlines()` читает оставшиеся строки в список. Символы `\n` сохраняются.

In [ ]:
with open(folder / "lines.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print(lines)

## 11. Запись списка через writelines()

`writelines()` записывает строки списка, но не добавляет разделители. Поэтому переводы строк включены в сами строки.

In [ ]:
lines = ["Один\n", "Два\n", "Три\n"]

with open(folder / "numbers.txt", "w", encoding="utf-8") as file:
    file.writelines(lines)

with open(folder / "numbers.txt", "r", encoding="utf-8") as file:
    print(file.read())

## 12. Удаление перевода строки

`rstrip("\n")` убирает перевод строки справа, сохраняя пробелы.

`strip()` убирает пробельные символы с обоих краёв, в том числе обычные пробелы.

In [ ]:
line = "  Текст с пробелами  \n"

print("Исходная строка:", repr(line))
print("Без перевода строки:", repr(line.rstrip("\n")))
print("После strip():", repr(line.strip()))

## 13. Чтение части файла

`read(5)` читает не более пяти символов. Следующее чтение продолжается с текущего места. `seek(0)` возвращает в начало.

In [ ]:
with open(folder / "numbers.txt", "r", encoding="utf-8") as file:
    print("Первые пять символов:", repr(file.read(5)))
    print("Остаток:", repr(file.read()))

    file.seek(0)
    print("Снова весь файл:", repr(file.read()))

## 14. Два файла: чтение и запись

Открываем источник на чтение, а новый файл — на запись. Оба файла закроются автоматически.

Для простоты используем два вложенных блока `with`. Источник и копия — разные файлы.

In [ ]:
with open(folder / "lines.txt", "r", encoding="utf-8") as source:
    with open(folder / "copy.txt", "w", encoding="utf-8") as target:
        for line in source:
            target.write(line)

with open(folder / "copy.txt", "r", encoding="utf-8") as file:
    print(file.read())

## 15. Бинарный файл

Режим `wb` — запись байтов, `rb` — чтение байтов. Для бинарного режима не указывают кодировку.

Здесь используем простую последовательность байтов, записанную с префиксом `b`.

In [ ]:
with open(folder / "data.bin", "wb") as file:
    file.write(b"Hello, Python!")

with open(folder / "data.bin", "rb") as file:
    data = file.read()

print(data)

In [ ]:
with open(folder / "data.bin", "wb") as file:
    file.write(b"Привет, Python!")

In [ ]:
with open(folder / "ex2.jpg", "wb") as file:
    file.write(b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x02\x02\x03\x02\x02\x02\x02\x02\x04\x03\x03\x02\x03\x05\x04\x05\x05\x05\x04\x04\x04\x05\x06\x07\x06\x05\x05\x07\x06\x04\x04\x06\t\x06\x07\x08\x08\x08\x08\x08\x05\x06\t\n\t\x08\n\x07\x08\x08\x08\xff\xdb\x00C\x01\x01\x01\x01\x02\x02\x02\x04\x02\x02\x04\x08\x05\x04\x05\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\xff\xc0\x00\x11\x08\x002\x002\x03\x01\x11\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1e\x00\x00\x01\x04\x02\x03\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x07\t\n\x06\x08\x02\x03\x04\x05\xff\xc4\x00-\x10\x00\x01\x03\x03\x04\x02\x01\x03\x03\x05\x01\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x11\x00\x07\x08\x12\x131!\x14"Q\x152A\t#3Ba\xb1\xff\xc4\x00\x1d\x01\x00\x01\x03\x05\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x05\x08\x02\x04\x06\x07\t\x03\xff\xc4\x005\x11\x00\x01\x02\x04\x05\x03\x01\x05\x06\x07\x00\x00\x00\x00\x00\x00\x01\x02\x11\x00\x03\x04\x05\x06\x12!1A\x07Qa"\x13\x142q\x81\x15BR\x91\xb1\xc1\x08\x16#3\xa1\xd1\xf0\xff\xda\x00\x0c\x03\x01\x00\x02\x11\x03\x11\x00?\x00\xbf\xc6\x88 \xc8\xfc\xe8\x82<\xb2\xdfLx\xef\xc8\xca0\x84)d\xab8\x18\x19\xf9 \x13\x8dy\xcdS\x070\x04\xb9hl6>\xfe\x9b\xb9\xdbUe_\xd54R\xd9\x9dS\x8a_u0\x83\x81\x84\x90\xe2\x93\x84\xf9\x00P\xfd\xbf9\xfes\x8c\x8c\x1d0a[\xdf\xda4\x12\xab4\xf5\x87\xd1\xdbr9\xd7\x88q\xbb\xd0\xfb\xb5J\xa4~\x1f\xf5\x0e\xceA\xf4A\xd6F\xf0\xdd\x06\x88 \xd1\x04\x1a\t\x82:\xdcPJI?\x03\xff\x005A"\x08\xacg+\xb7\xb7\x93\xf6\xd737^[\x97\x15Y\xa96\xb4\xb5\xbfh\xd1\xe4:\xf2i\xc8\x8c[\x0eE|GmIK\xa8X\t\x0e9\xf2\xa0\xa5\x91\x94\x8dA\xbe\xadc\x0b\xad& )\x9b9IL\xb5\x82\x90\xec\x9c\xbci\xcb\xf9\xde$\x16\x10\xb1\xd0\xcd\xb5\xa4\xa5 \x95\x86Q\xdc\xbf?\x94k%K{\xf9+G\xaam\xadb\xc0\xaf\xd6\xed\xdb\xb2\x9b16\xfd\xb0\xc5\r\xc7\x90*\x11|\xb9e\x97#)E\x12{\xb8\xeb\x9d\xd2S\xd7\xefG\xc2I\xd6\x11\x86q\x9d\xcf\xdf\xe4\x8a\x19\xaa\xcc\x94\xa5 \x07c\xa9,\xdbjN\xb1\x91WXhL\x89\x9e\xdd\x00\x82\xea$\xee4g\x07\xe9\x17\x00\xa4.{\x94\xe8*\xaa!\x86\xea\x9e\x16\xfe\xa5-\x1c\xa1/u\x1d\xc2O\xe0+:\xe8\xbc\x95+ +\xdf\x9f\x9f1\x17V\x03\x9c\xbbG\xd2\xd7\xb01L\x1aX!\x0f\xafY\xd2\x18\x0cCG\xf5\x03\xe5\xbdv4\x8d\xca\xe3e\xabK\x9bI\x9c\xd7\xe9e\xda\xc33|Jy\xb5\'\xce\xf4p\x9cv\x05I,\xa5%>\xf2\xb0}\x8dA\xef\xe2G\xae\x13)\x97Q\x85iR\xa4M!\r0\x1e\x0f\xa9A\xb7\xd7@\x1br\xf1gSP\x90\xe8<\xf8\x86\x86\xd2\xe3^\xda\xee%\xe9e\xdb[\xff\x00\xb8\xb7\xdc]\xf2\x98\xca\xa3Lz\x1dAU\x06P\xfa\x94\xa7\x18\x8b!\xf7\xf3\xe2\x94\x10\xb5\'\xc6\xdaJQ\xe9K\n_P\xcb\x850]\x1dt\xda{>$\xae_\xdaK\x05D\x03\x9f}R\x95(\xfc*\xcb\xf7\x06\x83\xbb\x98\xda\xd8J\xba\xe9n\xb5*\xa52\x82\x90K\xb9%\xfbh;C\x8f\xbb\x9c\r\xd8}\x96\x99h]\x12/-\xc15\x97\x9ey\x8a<\xb9N}<\x083\xd2\xc2\x8b\x0eJ\x97\x1d!q\xd2\x95\xa8)*\t^\n\t#\xa8Q\x19\xbe)\xe8\xed\xaa\xc3(MER\xd0\xb9\x99\x92\x85\x1fJR\xb2=9\x96\x90\xe8s\xb2\xbb\xe9\xb4{\xcf\xc5w\x0b\xbd$\xeaYh\x0f\x95\xf4$)\xbc\x0e|\xc3[\xc6\xfeS\xeeW\x1a\xf7\x06\xf8\xdb\r\xd9\x8bY\xbc\xe7\xd4\xee\x8at7\xd7"\xaed\x1ak\x9eO\x04\x87\xc1VK\x9d\xd0Y( \xe1]A\xf5\x83\xadC\xd2\xae\xb5\\\xb0\xc5\xcau\x8e\xfd\x9e|\xe9\x93R\x92\xea\'!|\xaa/\xce`A\r\xa1g\x8d="~S\xec\xd7\xaa\xa2\xc0\t9\xcf\xfc8\xd7H\x848B\xe9`\x8e+ $\x93\x80\x07\xc9\xd2*\x08\xae%\x97c\xdf[\x93\xccW\xe8\xdb\x85{\xc0\x8fwF\x9d"\x9d\x16\xe0\x96\x96\xa5\x87\xd4\x96]r\x08\x8f\x8c\xa5\xf2[\xea\xa4\xb8~\xe4$\x0f\xb9*JF\xb9\x83o\xb1\xdd\xaex\xf7\xddn\xf5\x08\xcc\x95\x14\xa6a\xca\xa2tQ\x96\x13\xf8\x8fg\xd9\xb5\xd4E\x14sL\xba\xa4\xaap\nBH-\xdcn\xd1\xbd\xb4\xbd\x89\xdc\x8e0\xb1F\xdf\x97\x1e\xb7\xaf\x99t\xa4\xcar\xe1\xa56\t\xfax+\xff\x00#\xf1\xa49\x85-\xd0\x9c\xadJ)\n\xf9 v\x05y\x906\x1e\x94\xdcpz\x86#\x0b\x15SFol\x9er\x92\xe5HQ\xd70\x1a\x97\xdfh\xdb\xd5\xf8\xaa\x9e\xee\r\xb9\xbd\x92Kd>x\n\x03\x83\xc7hs\xa1\xd5j\\\xd5\xadt\x88\x8a\xa5\xb9\xc7JD\xa4\xfd@Q\xf1M\xb8\xa7\xa0\x05x\x97\x82T\xc2\x1b\xec\t\x1f\xb8\x12?\xd8\x8f\x1et\x99\xff\x00\xcf\x8a(H)\xb5 \x87;\x19\xca\x1a\xb7t\xa1\'~I\x86eJ\xfb\x04f,j\x94>a\x03\xbf\x92x\x88\xd1\xe6v\xdb\\\xdb=\xba6\x81\xa4H\x8dW\x9dDm\x87m\xd9HKnT>\x94\xc8&\x1a\xa4 \x8c:\xa6]\x1fN\x95,\x11\xd5\xa6N\x00\xf5\x14\xfa\xfb\x86\xeb,\xd7\xbaU\xdb\x96\x95M@NR[\xda0SK*\x7f\x89\x8f\xa31\xe0\x07\x8c\x1e\xfdT\'\xcfML\x80\x02\xd5\xf1\r\xb5\xd9\xc7\x83\xfa\xc4\xf0\xec\xabW\x13\x1bK\xb7L]\xd5\xa6\xee+\xa9\x14x\xa8\xa9ML\x94\xc8\x12%\x06\xc7\x94\xf9RHY\xed\xd8\x15\x02rA\xd7A\xb0<\xba\xa4\xda)\x93[3\xdaN\x08Ne\x02\xee\xa6\xd4\xb8\xdfX\xb4H`\xdb\xc3\x9f\xac\xaa\x16\x11Y\xc7\xc7\xbd!\xda\x08\xae\x97"\xac\xfbk\x88\\\x9a\x87zm\x9d\xd3n\\\x8e!\xc9\xb5\xa8\x94E\xf4S\x96\xf5Ac!\xa9=U\x957\xfd\xe0\xe2\x12B\x14@\xc18\x1d\x8f0\xfa\xb7`\xa0\xc0\xd8\xae]\xda\xd8\xb15d\xa9i\x96H\xfe\x9a\xf8\xcc\x06\xb9K\x96\xd8\xf1\r\x93\xd1.Z\x82\xc0u~\x9f>Z$\x0e\xde\xaf]\xfc\xe1]\xaa\x0c\xa8\x16\xc6\xc6\xc7\x87\x12}e\x9amA.\xc8\x9d8\x8c\xaa#\xb8=\xdaR\x16\x95a*\x00\x04up\x15\xa9H\xf1\xca\x0bm\xca\xab\xa8\x12\xe4\x94\xa8K\xb6\xe5J\xa6\x04\xa9\xd6\xa5\xf3-M\xaa@#W\xdcF\xc2\xb1]\xe8\xe8\xe9=\xea_\xaa\xa5N\x03\xec\x8f>Oh\xc876\xc2\xaeqr\xe1ox6I\x8a]/o\xe4-\xa6.Ky\xf7\xd3\x1e\x9e\xd0\xce\x03\xe1J 6\x93\xf3\x95\x8c\xa9\x0bV@ZV\xa4i\xcf\x14\xd9*p\xa5X\xbc\xd8\xf2\xa6\x95D\t\xf2\x94\xac\xa8\x03a1$\xfc$s\xdf\xb4^S\xde\xe9\xab\xe9\x95OtS-.R\xbe|\xa4\xf7\xf1\x11s{_\xf09\xbd\xc8;]\x8b\xb6\xe1\xb6v\x8a\x94\xe3\xd2\xe9\xbf\xa9>\xb4\xa9\xb4R\xd2\xae\xcc\xb7\x95\x94\x83$\x92\xe2\x02\xc9\tQy?h\t\xc2\xa1\x9d\xdf\x10\xd2u\x1b\x16IMr\x85*ARB\xdc\x7fl\x17@\xd7\xef\x92O\x8d|F\xae\x05\x13\xa6z\x92\xcd\xce\xdf/\xacX\xa6\xca\xb4\xa8\xb6\x15\xa5nYV\xe4s\x12\x85J\x86\xd4\x08\x8d\x92\tKM\xa4$v#\x19Q\xc6I\xfeI:\xe9\xfd\x82\xcb"\xddE*\x82\x944\xb9I\t\x1f \x1b\xf3\xef\x0e\x89\x00\x06\x11\x94i\xde*\x83Hv\x8253\x93<]\xda\xdd\xeb\xb6n:\xd5[mb\xdc\xdb\x8a\xdc\x05\x8ad\xa8\xb2\x13\n[\x92\x12\x0f\x88\x17\x94Cj\xc1\xf5\xe5\nH\x19\x1a\xd3}P\xe9\x1d\x9a\xfbO6\xa6}(\x99T\x10BT\x93\x91gM\x06m\xb7\xdb3\x88\xf2\\\xa4\x92\xed\xaf\xfd\xccC\x96\xddq?\x9c;ws\xbb.\xdd\xb0\xae\xfbN\x93"\n\xdby\xca5\xc4\xc4R\xeb\xc9q\x1d\x12\xe3m\xc8FS\xd4\xb8RI#\xde\xa0\xd6\x1b\xe8\x97Qm2\x956\xd6\x95\xcaZ\xcb\x10&$\x16\x1b\x12\xc5\x8f\x8f\xda=\xedr\x11)%su\'F:\xfd\x7fm\xe3!\xdc\xfe6\xf3\xca\xfe\x8d@\xa3\xd4\xedm\xc3\xbaih\x92\xb9\x12\xd8\xaa],\xbb\x1d\x01,\x92\x95x\xdc\x92\xb0\xa5\xf7\x18H\xc6~\xef~\xf5}y\xe9\x07S\xae\xb4\xab\xa7\xba\x95\xccJYA&` \x91\xe1\xd9\xfb~\xc6*\xa9BgJP\xd1*\x0cC>\xbc\x11\xfe_\xe9\xde7\xdf\x86\x1c-\xb2\xf6\xfe\xd2\xa5]\xfb\xa5\xb4Q)\xdb\xc8\xd4\xb5\xadON\x94\xdc\xa46\x84\x9c\xb4\xb6YmE\xa6\xfa\x85\x1cv\nXVOo\x90\x04\x91\xe8gAm\xf6\xdb|\x8a\xcb\xc5\x13W$\x92\xebP[\x1e\x08\x00\xe5\x1e7#\xbcYH\xa7HH\n\x1a\x8e\xfa\xc4\x94jT\xa6.`\xd5PA\xa2\x084A\t\x81\xf8\x1aF\x10A\x81\xf8\x1a\x1a\x08],\x10h\x82\r\x10G\xff\xd9')

In [ ]:
with open(folder / "ex3.jpg", "wb") as file:
    file.write(b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x02\x02\x03\x02\x02\x02\x02\x02\x04\x03\x03\x02\x03\x05\x04\x05\x05\x05\x04\x04\x04\x05\x06\x07\x06\x05\x05\x07\x06\x04\x04\x06\t\x06\x07\x08\x08\x08\x08\x08\x05\x06\t\n\t\x08\n\x07\x08\x08\x08\xff\xdb\x00C\x01\x01\x01\x01\x02\x02\x02\x04\x02\x02\x04\x08\x05\x04\x05\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\x08\xff\xc0\x00\x11\x08\x002\x002\x03\x01\x11\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1e\x00\x00\x01\x04\x02\x03\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x07\t\n\x06\x08\x02\x03\x04\x05\xff\xc4\x00-\x10\x00\x01\x03\x03\x04\x02\x01\x03\x03\x05\x01\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x11\x00\x07\x08\x12\x131!\x14"Q\x152A\t#3Ba\xb1\xff\xc4\x00\x1d\x01\x00\x01\x03\x05\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x05\x08\x02\x04\x06\x07\t\x03\xff\xc4\x005\x11\x00\x01\x02\x04\x05\x03\x01\x05\x06\x07\x00\x00\x00\x00\x00\x00\x01\x02\x11\x00\x03\x04\x05\x06\x12!1A\x07Qa"\x13\x142q\x81\x15BR\x91\xb1\xc1\x08\x16#3\xa1\xd1\xf0\xff\xda\x00\x0c\x03\x01\x00\x02\x11\x03\x11\x00?\x00\xbf\xc6\x88 \xc8\xfc\xe8\x82<\xb2\xdfLx\xef\xc8\xca0\x84)d\xab8\x18\x19\xf9 \x13\x8dy\xcdS\x070\x04\xb9hl6>\xfe\x9b\xb9\xdbUe_\xd54R\xd9\x9dS\x8a_u0\x83\x81\x84\x90\xe2\x93\x84\xf9\x00P\xfd\xbf9\xfes\x8c\x8c\x1d0a[\xdf\xda4\x12\xab4\xf5\x87\xd1\xdbr9\xd7\x88q\xbb\xd0\xfb\xb5J\xa4~\x1f\xf5\x0e\xceA\xf4A\xd6F\xf0\xdd\x06\x88 \xd1\x04\x1a\t\x82:\xdcPJI?\x03\xff\x005A"\x08\xacg+\xb7\xb7\x93\xf6\xd737^[\x97\x15Y\xa96\xb4\xb5\xbfh\xd1\xe4:\xf2i\xc8\x8c[\x0eE|GmIK\xa8X\t\x0e9\xf2\xa0\xa5\x91\x94\x8dA\xbe\xadc\x0b\xad& )\x9b9IL\xb5\x82\x90\xec\x9c\xbci\xcb\xf9\xde$\x16\x10\xb1\xd0\xcd\xb5\xa4\xa5 \x95\x86Q\xdc\xbf?\x94k%K{\xf9+G\xaam\xadb\xc0\xaf\xd6\xed\xdb\xb2\x9b16\xfd\xb0\xc5\r\xc7\x90*\x11|\xb9e\x97#)E\x12{\xb8\xeb\x9d\xd2S\xd7\xefG\xc2I\xd6\x11\x86q\x9d\xcf\xdf\xe4\x8a\x19\xaa\xcc\x94\xa5 \x07c\xa9,\xdbjN\xb1\x91WXhL\x89\x9e\xdd\x00\x82\xea$\xee4g\x07\xe9\x17\x00\xa4.{\x94\xe8*\xaa!\x86\xea\x9e\x16\xfe\xa5-\x1c\xa1/u\x1d\xc2O\xe0+:\xe8\xbc\x95+ +\xdf\x9f\x9f1\x17V\x03\x9c\xbbG\xd2\xd7\xb01L\x1aX!\x0f\xafY\xd2\x18\x0cCG\xf5\x03\xe5\xbdv4\x8d\xca\xe3e\xabK\x9bI\x9c\xd7\xe9e\xda\xc33|Jy\xb5\'\xce\xf4p\x9cv\x05I,\xa5%>\xf2\xb0}\x8dA\xef\xe2G\xae\x13)\x97Q\x85iR\xa4M!\r0\x1e\x0f\xa9A\xb7\xd7@\x1br\xf1gSP\x90\xe8<\xf8\x86\x86\xd2\xe3^\xda\xee%\xe9e\xdb[\xff\x00\xb8\xb7\xdc]\xf2\x98\xca\xa3Lz\x1dAU\x06P\xfa\x94\xa7\x18\x8b!\xf7\xf3\xe2\x94\x10\xb5\'\xc6\xdaJQ\xe9K\n_P\xcb\x850]\x1dt\xda{>$\xae_\xdaK\x05D\x03\x9f}R\x95(\xfc*\xcb\xf7\x06\x83\xbb\x98\xda\xd8J\xba\xe9n\xb5*\xa52\x82\x90K\xb9%\xfbh;C\x8f\xbb\x9c\r\xd8}\x96\x99h]\x12/-\xc15\x97\x9ey\x8a<\xb9N}<\x083\xd2\xc2\x8b\x0eJ\x97\x1d!q\xd2\x95\xa8)*\t^\n\t#\xa8Q\x19\xbe)\xe8\xed\xaa\xc3(MER\xd0\xb9\x99\x92\x85\x1fJR\xb2=9\x96\x90\xe8s\xb2\xbb\xe9\xb4{\xcf\xc5w\x0b\xbd$\xeaYh\x0f\x95\xf4$)\xbc\x0e|\xc3[\xc6\xfeS\xeeW\x1a\xf7\x06\xf8\xdb\r\xd9\x8bY\xbc\xe7\xd4\xee\x8at7\xd7"\xaed\x1ak\x9eO\x04\x87\xc1VK\x9d\xd0Y( \xe1]A\xf5\x83\xadC\xd2\xae\xb5\\\xb0\xc5\xcau\x8e\xfd\x9e|\xe9\x93R\x92\xea\'!|\xaa/\xce`A\r\xa1g\x8d="~S\xec\xd7\xaa\xa2\xc0\t9\xcf\xfc8\xd7H\x848B\xe9`\x8e+ $\x93\x80\x07\xc9\xd2*\x08\xae%\x97c\xdf[\x93\xccW\xe8\xdb\x85{\xc0\x8fwF\x9d"\x9d\x16\xe0\x96\x96\xa5\x87\xd4\x96]r\x08\x8f\x8c\xa5\xf2[\xea\xa4\xb8~\xe4$\x0f\xb9*JF\xb9\x83o\xb1\xdd\xaex\xf7\xddn\xf5\x08\xcc\x95\x14\xa6a\xca\xa2tQ\x96\x13\xf8\x8fg\xd9\xb5\xd4E\x14sL\xba\xa4\xaap\nBH-\xdcn\xd1\xbd\xb4\xbd\x89\xdc\x8e0\xb1F\xdf\x97\x1e\xb7\xaf\x99t\xa4\xcar\xe1\xa56\t\xfax+\xff\x00#\xf1\xa49\x85-\xd0\x9c\xadJ)\n\xf9 v\x05y\x906\x1e\x94\xdcpz\x86#\x0b\x15SFol\x9er\x92\xe5HQ\xd70\x1a\x97\xdfh\xdb\xd5\xf8\xaa\x9e\xee\r\xb9\xbd\x92Kd>x\n\x03\x83\xc7hs\xa1\xd5j\\\xd5\xadt\x88\x8a\xa5\xb9\xc7JD\xa4\xfd@Q\xf1M\xb8\xa7\xa0\x05x\x97\x82T\xc2\x1b\xec\t\x1f\xb8\x12?\xd8\x8f\x1et\x99\xff\x00\xcf\x8a(H)\xb5 \x87;\x19\xca\x1a\xb7t\xa1\'~I\x86eJ\xfb\x04f,j\x94>a\x03\xbf\x92x\x88\xd1\xe6v\xdb\\\xdb=\xba6\x81\xa4H\x8dW\x9dDm\x87m\xd9HKnT>\x94\xc8&\x1a\xa4 \x8c:\xa6]\x1fN\x95,\x11\xd5\xa6N\x00\xf5\x14\xfa\xfb\x86\xeb,\xd7\xbaU\xdb\x96\x95M@NR[')
    

In [ ]:
with open(folder / "ex3.jpg", "ab") as file:
    file.write(b'\xda0SK*\x7f\x89\x8f\xa31\xe0\x07\x8c\x1e\xfdT\'\xcfML\x80\x02\xd5\xf1\r\xb5\xd9\xc7\x83\xfa\xc4\xf0\xec\xabW\x13\x1bK\xb7L]\xd5\xa6\xee+\xa9\x14x\xa8\xa9ML\x94\xc8\x12%\x06\xc7\x94\xf9RHY\xed\xd8\x15\x02rA\xd7A\xb0<\xba\xa4\xda)\x93[3\xdaN\x08Ne\x02\xee\xa6\xd4\xb8\xdfX\xb4H`\xdb\xc3\x9f\xac\xaa\x16\x11Y\xc7\xc7\xbd!\xda\x08\xae\x97"\xac\xfbk\x88\\\x9a\x87zm\x9d\xd3n\\\x8e!\xc9\xb5\xa8\x94E\xf4S\x96\xf5Ac!\xa9=U\x957\xfd\xe0\xe2\x12B\x14@\xc18\x1d\x8f0\xfa\xb7`\xa0\xc0\xd8\xae]\xda\xd8\xb15d\xa9i\x96H\xfe\x9a\xf8\xcc\x06\xb9K\x96\xd8\xf1\r\x93\xd1.Z\x82\xc0u~\x9f>Z$\x0e\xde\xaf]\xfc\xe1]\xaa\x0c\xa8\x16\xc6\xc6\xc7\x87\x12}e\x9amA.\xc8\x9d8\x8c\xaa#\xb8=\xdaR\x16\x95a*\x00\x04up\x15\xa9H\xf1\xca\x0bm\xca\xab\xa8\x12\xe4\x94\xa8K\xb6\xe5J\xa6\x04\xa9\xd6\xa5\xf3-M\xaa@#W\xdcF\xc2\xb1]\xe8\xe8\xe9=\xea_\xaa\xa5N\x03\xec\x8f>Oh\xc876\xc2\xaeqr\xe1ox6I\x8a]/o\xe4-\xa6.Ky\xf7\xd3\x1e\x9e\xd0\xce\x03\xe1J 6\x93\xf3\x95\x8c\xa9\x0bV@ZV\xa4i\xcf\x14\xd9*p\xa5X\xbc\xd8\xf2\xa6\x95D\t\xf2\x94\xac\xa8\x03a1$\xfc$s\xdf\xb4^S\xde\xe9\xab\xe9\x95OtS-.R\xbe|\xa4\xf7\xf1\x11s{_\xf09\xbd\xc8;]\x8b\xb6\xe1\xb6v\x8a\x94\xe3\xd2\xe9\xbf\xa9>\xb4\xa9\xb4R\xd2\xae\xcc\xb7\x95\x94\x83$\x92\xe2\x02\xc9\tQy?h\t\xc2\xa1\x9d\xdf\x10\xd2u\x1b\x16IMr\x85*ARB\xdc\x7fl\x17@\xd7\xef\x92O\x8d|F\xae\x05\x13\xa6z\x92\xcd\xce\xdf/\xacX\xa6\xca\xb4\xa8\xb6\x15\xa5nYV\xe4s\x12\x85J\x86\xd4\x08\x8d\x92\tKM\xa4$v#\x19Q\xc6I\xfeI:\xe9\xfd\x82\xcb"\xddE*\x82\x944\xb9I\t\x1f \x1b\xf3\xef\x0e\x89\x00\x06\x11\x94i\xde*\x83Hv\x8253\x93<]\xda\xdd\xeb\xb6n:\xd5[mb\xdc\xdb\x8a\xdc\x05\x8ad\xa8\xb2\x13\n[\x92\x12\x0f\x88\x17\x94Cj\xc1\xf5\xe5\nH\x19\x1a\xd3}P\xe9\x1d\x9a\xfbO6\xa6}(\x99T\x10BT\x93\x91gM\x06m\xb7\xdb3\x88\xf2\\\xa4\x92\xed\xaf\xfd\xccC\x96\xddq?\x9c;ws\xbb.\xdd\xb0\xae\xfbN\x93"\n\xdby\xca5\xc4\xc4R\xeb\xc9q\x1d\x12\xe3m\xc8FS\xd4\xb8RI#\xde\xa0\xd6\x1b\xe8\x97Qm2\x956\xd6\x95\xcaZ\xcb\x10&$\x16\x1b\x12\xc5\x8f\x8f\xda=\xedr\x11)%su\'F:\xfd\x7fm\xe3!\xdc\xfe6\xf3\xca\xfe\x8d@\xa3\xd4\xedm\xc3\xbaih\x92\xb9\x12\xd8\xaa],\xbb\x1d\x01,\x92\x95x\xdc\x92\xb0\xa5\xf7\x18H\xc6~\xef~\xf5}y\xe9\x07S\xae\xb4\xab\xa7\xba\x95\xccJYA&` \x91\xe1\xd9\xfb~\xc6*\xa9BgJP\xd1*\x0cC>\xbc\x11\xfe_\xe9\xde7\xdf\x86\x1c-\xb2\xf6\xfe\xd2\xa5]\xfb\xa5\xb4Q)\xdb\xc8\xd4\xb5\xadON\x94\xdc\xa46\x84\x9c\xb4\xb6YmE\xa6\xfa\x85\x1cv\nXVOo\x90\x04\x91\xe8gAm\xf6\xdb|\x8a\xcb\xc5\x13W$\x92\xebP[\x1e\x08\x00\xe5\x1e7#\xbcYH\xa7HH\n\x1a\x8e\xfa\xc4\x94jT\xa6.`\xd5PA\xa2\x084A\t\x81\xf8\x1aF\x10A\x81\xf8\x1a\x1a\x08],\x10h\x82\r\x10G\xff\xd9')


## Краткая памятка

| Что нужно | Режим |
|---|---|
| Прочитать текст | `r` |
| Записать текст заново | `w` |
| Добавить текст в конец | `a` |
| Прочитать байты | `rb` |
| Записать байты заново | `wb` |

Стандартные шаблоны работы:

```python
with open(folder / "example.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text)
```
---
```python
lines = []
with open(folder / "example.txt", "r", encoding="utf-8") as file:
    for line in file:
        lines.append(line)

print(lines)
```
---
```python
lines = ["Первая строка", "Вторая строка", "Третья строка"]
with open(folder / "example.txt", "ц", encoding="utf-8") as file:
    for line in lines:
        file.write(f"{line}\n")
```

---
---